# 14.10 · 时序异常检测 / Time Series Anomaly Detection

> **课程定位 / Where this fits**
> 第 10 课，**Part 14 · 时间序列**。在序列里找"不正常"的点。
> Lesson 10, **Part 14 · Time Series**. Finding "abnormal" points in a series.
>
> 时序异常检测无处不在: 服务器监控(突发故障)、金融(欺诈交易)、设备(故障预警)、网站(流量异常)。核心思路出奇统一: **先建模"正常应该是什么样", 再看实际偏离正常多少, 偏太多就是异常**。怎么建"正常"? 用前面学的**分解/预测**(STL、ARIMA…)得到期望值, 再看**残差**(实际−期望)。本课用 STL 残差 + **稳健统计(MAD)** 检测注入的异常, 并讲清异常类型和评估。
> TS anomaly detection is everywhere: server monitoring (sudden failures), finance (fraud), equipment (fault alerts), web (traffic spikes). The core idea is strikingly uniform: **model "what's normal," then see how far reality deviates; too far = anomaly**. How to model "normal"? Use the **decomposition/forecasting** we learned (STL, ARIMA…) to get expected values, then watch the **residual** (actual − expected). We detect injected anomalies using STL residuals + **robust statistics (MAD)** and cover anomaly types and evaluation.
>
> 💼 **实战/面试视角**："异常检测的通用思路(建模正常+看残差) / 为什么用稳健统计(MAD vs z-score) / 异常类型 / 怎么评估" 是监控/风控岗常考。
> 💼 **Practical/interview angle:** "the general approach (model normal + residual) / why robust stats (MAD vs z-score) / anomaly types / evaluation" — monitoring/risk roles.

> 📐 **符号约定 / Notation**
> - 残差 —— 实际值 − 模型期望值(正常时应小) / actual − expected (small when normal)
> - MAD —— 中位数绝对偏差(稳健的"标准差") / median absolute deviation (robust spread)

> 💡 **面试相关 / Interview-relevant**
> - "时序异常检测的通用框架"（出镜率 ★★★★）
> - "为什么用 MAD/中位数而非均值/标准差(稳健性)"（出镜率 ★★★★★）
> - "异常类型(点/上下文/集体)"（★★★★）
> - "异常检测怎么评估(precision/recall, 不平衡)"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解异常检测的通用框架: 建模正常 → 看残差。
   Understand the general framework: model normal → watch residuals.
2. 用 **STL 残差** 检测异常。
   Detect anomalies via STL residuals.
3. 理解为何用**稳健统计(MAD)** 而非普通 z-score。
   Understand why robust statistics (MAD) over plain z-score.
4. 用 precision/recall 评估, 了解异常类型。
   Evaluate with precision/recall; know anomaly types.

## 目录 / TOC
1. [异常检测框架 + 异常类型 ⭐](#1)
2. [STL 残差 + z-score 检测 ⭐](#2)
3. [稳健统计:MAD ⭐](#3)
4. [评估 + 其他方法 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 异常检测框架 + 异常类型 ⭐ / Framework & Anomaly Types

时序异常检测的**通用框架**(面试):
The **general framework** for TS anomaly detection (interview):
1. **建模"正常"**:用分解(14.2)或预测模型(14.3-14.8)算出每个时刻**期望的正常值**。
   **Model "normal":** use decomposition (14.2) or a forecast model (14.3-14.8) to get the **expected normal value** at each time.
2. **算残差**:残差 = 实际值 − 期望值。正常时残差是小的随机波动; 异常时残差**突然变大**。
   **Compute residuals:** residual = actual − expected. Normally small noise; an anomaly makes it **suddenly large**.
3. **定阈值**:残差超过某阈值就标为异常。
   **Threshold:** flag points whose residual exceeds a threshold.

**异常的三种类型**(面试常考)：
**Three anomaly types** (interview):
- **点异常(point)**:单个点突然偏离(如某秒流量暴涨)。最常见, 最易检测。
  **Point:** a single point deviates (a traffic spike). Most common, easiest.
- **上下文异常(contextual)**:值本身不极端, 但在**当时的上下文**里反常(如冬天的"夏季高温")。
  **Contextual:** the value isn't extreme per se but is abnormal **in context** (summer-level heat in winter).
- **集体异常(collective)**:一段子序列整体反常(单看每个点都正常, 但连在一起的模式异常)。
  **Collective:** a whole subsequence is abnormal (each point fine alone, but the pattern is off).

本课聚焦最常见的**点异常**, 用 STL 残差检测。先造一个带注入异常的序列(已知真值, 便于评估)。
We focus on the common **point anomalies** via STL residuals. First, build a series with injected anomalies (known ground truth for evaluation).


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
from statsmodels.tsa.seasonal import STL
sns.set_theme(style="whitegrid"); np.random.seed(0)

# 合成: 趋势+季节+噪声 的正常序列, 再注入【较多】点异常(已知位置=真值) / synthetic series + MANY injected anomalies
n = 400
t = np.arange(n)
normal = 50 + 0.1*t + 10*np.sin(2*np.pi*t/30) + np.random.randn(n)*1.5   # 趋势+季节(周期30)+噪声 / trend+season+noise
series = normal.copy()
true_anom = np.sort(np.random.choice(np.arange(10, n-10), 30, replace=False))   # 注入30个异常(故意多) / 30 anomalies
series[true_anom] += np.random.choice([-1,1], 30) * (15 + np.random.rand(30)*8)  # 较大偏移 / large offsets
idx = pd.date_range("2020-01-01", periods=n, freq="D"); ts = pd.Series(series, index=idx)

fig, ax = plt.subplots(figsize=(12, 4)); ts.plot(ax=ax, label="序列")
ax.plot(idx[true_anom], ts.iloc[true_anom], "rX", ms=9, label="注入的真实异常")
ax.legend(); ax.set_title(f"带注入异常的序列: {len(true_anom)}个点偏离了'正常的趋势+季节'模式")
plt.tight_layout(); plt.show()
print(f"合成序列: 趋势+季节(周期30)+噪声, 注入 {len(true_anom)} 个点异常(故意注入较多, 为揭示下面普通z-score的缺陷)")
print("通用框架: 建模正常(分解/预测) → 算残差(实际-期望) → 残差超阈值=异常")


<a id="2"></a>
## 2. STL 残差 + z-score 检测 ⭐ / STL Residual + Z-Score

用 **STL 分解**(14.2)建模"正常": 趋势 + 季节就是序列**正常应有**的形状, **残差**就是去掉它们后的偏差。在正常点, 残差是小噪声; 在异常点, 残差**很大**。
Use **STL decomposition** (14.2) to model "normal": trend + seasonal is the **expected** shape, and the **residual** is the deviation after removing them. At normal points the residual is small noise; at anomalies it's **large**.

最朴素的阈值: **z-score**——残差减均值除以标准差, $|z| > 3$ 就算异常(超出 3 个标准差)。
The naive threshold: **z-score** — residual minus mean over std; $|z| > 3$ flags an anomaly (beyond 3 sigma).


In [ ]:
stl = STL(ts, period=30, robust=True).fit()
resid = stl.resid                                         # STL 残差 = 去掉趋势季节后的偏差 / residual
TH = 3.5                                                  # 阈值 / threshold
z = (resid - resid.mean()) / resid.std()                  # 普通 z-score: 用均值/标准差 / plain z-score
detected_z = np.where(np.abs(z) > TH)[0]                  # |z|>3.5 标为异常 / flag

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ts.plot(ax=axes[0], label="序列"); axes[0].plot(idx[true_anom], ts.iloc[true_anom], "gX", ms=9, label="真实异常")
axes[0].legend(); axes[0].set_title("序列 + 真实异常")
axes[1].plot(idx, resid, label="STL 残差"); axes[1].axhline(0, color="gray", ls="--")
axes[1].plot(idx[detected_z], resid.iloc[detected_z], "ro", ms=8, mfc="none", label=f"普通z-score检测(|z|>{TH})")
axes[1].legend(); axes[1].set_title("STL残差 + 普通z-score检测")
plt.tight_layout(); plt.show()
def prf(detected, true, tol=2):
    tp = sum(any(abs(d-a)<=tol for a in true) for d in detected)
    prec = tp/len(detected) if len(detected) else 0
    rec = sum(any(abs(a-d)<=tol for d in detected) for a in true)/len(true); return prec, rec
pz, rz = prf(detected_z, true_anom)
print(f"残差标准差 = {resid.std():.1f}  ← 注意: 被这{len(true_anom)}个异常自己'抬高'了!")
print(f"普通z-score(|z|>{TH}): 检出{len(detected_z)}个; precision={pz:.2f}, recall={rz:.2f}")
print(f"问题: recall只有{rz:.2f} → 漏掉了不少真异常! 因为标准差被异常抬高→阈值变松→中等异常溜过去了")


<a id="3"></a>
## 3. 稳健统计:MAD ⭐ / Robust Statistics: MAD

普通 z-score 有个**致命弱点**(面试核心): 它用**均值和标准差**, 而**均值和标准差本身会被异常值严重拉偏**! 几个大异常会**抬高标准差**, 反而让阈值变松, 导致**漏检**(异常把检测异常的标尺给"污染"了)。
Plain z-score has a **fatal weakness** (interview core): it uses the **mean and std**, which are **themselves heavily skewed by the very anomalies** you're hunting! A few big anomalies **inflate the std**, loosening the threshold and causing **missed detections** (the anomalies corrupt the yardstick used to find them).

**稳健统计(robust statistics)** 用对异常**不敏感**的量替代:
**Robust statistics** use anomaly-insensitive quantities:
- 用**中位数(median)** 代替均值(中位数不受少数极端值影响)。
  Use the **median** instead of the mean (unaffected by a few extremes).
- 用 **MAD(中位数绝对偏差)** 代替标准差: MAD = median(|残差 − median(残差)|)。
  Use **MAD (median absolute deviation)** instead of std: MAD = median(|residual − median|).
- **稳健 z-score** = $0.6745 \times \dfrac{残差 - \text{median}}{\text{MAD}}$(0.6745 让它在正态下与普通 z 可比)。
  **Robust z-score** = $0.6745 \times \dfrac{\text{residual} - \text{median}}{\text{MAD}}$ (0.6745 makes it comparable under normality).

这正是 **Twitter 的 ESD(Generalized ESD)** 等经典时序异常检测的基础思想。下面对比普通 vs 稳健。
This underlies Twitter's **Generalized ESD** and other classic TS anomaly methods. Let's compare plain vs robust.


In [ ]:
med = resid.median()
mad = (resid - med).abs().median()                       # MAD: 中位数绝对偏差 / median absolute deviation
robust_z = 0.6745 * (resid - med) / mad                  # 稳健 z-score / robust z-score
detected_robust = np.where(np.abs(robust_z) > TH)[0]     # 同样阈值 / same threshold
pr, rr = prf(detected_robust, true_anom)
print(f"普通z-score: 尺度=标准差{resid.std():.1f}(被异常抬高); precision={pz:.2f}, recall={rz:.2f}")
print(f"稳健(MAD):    尺度=MAD{mad:.2f}(不受异常影响); precision={pr:.2f}, recall={rr:.2f}")
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(idx, ts, alpha=0.4, label="序列")
ax.plot(idx[true_anom], ts.iloc[true_anom], "gX", ms=11, label="真实异常")
ax.plot(idx[detected_robust], ts.iloc[detected_robust], "ro", ms=8, mfc="none", label="稳健MAD检测")
ax.legend(); ax.set_title(f"稳健(MAD)检测: recall={rr:.2f} (比普通z-score的{rz:.2f}高很多, 漏检少)")
plt.tight_layout(); plt.show()
print(f"\n关键洞察(面试核心): 普通z-score的标准差被异常自己拉大({resid.std():.1f}) → 阈值变松 → 漏检(recall仅{rz:.2f})")
print(f"MAD用中位数, 不受异常影响 → 尺度稳定({mad:.2f}) → 抓住更多真异常(recall {rr:.2f})")
print(f"诚实权衡: MAD recall高但 precision({pr:.2f})略低于普通z-score({pz:.2f}) —— 它更敏感, 多抓也多误报一点")
print("实务: 异常较多/较杂时, 稳健统计(MAD)的'不被异常污染'优势明显; 阈值再按precision/recall权衡调")


<a id="4"></a>
## 4. 评估 + 其他方法 + 小结 ⭐ / Evaluation & Other Methods

**怎么评估异常检测(面试)**: 异常通常**极少**(类别极不平衡), 所以**不能用准确率**(全判"正常"就有 99% 准确率却没用)。要用:
**How to evaluate (interview):** anomalies are usually **rare** (extreme imbalance), so **don't use accuracy** (predicting "all normal" scores 99% yet useless). Use:
- **Precision(精确率)**: 报警里有多少是真异常(别老误报, 否则"狼来了")。
  **Precision:** of the alerts, how many are real (avoid alert fatigue).
- **Recall(召回率)**: 真异常里抓住了多少(别漏掉重要故障)。
  **Recall:** of real anomalies, how many caught (don't miss critical faults).
- 实务要在**误报(precision)和漏检(recall)间权衡**——监控系统漏检故障的代价 vs 频繁误报的代价。还要考虑**检测延迟**(多快报警)。
  Trade off **false alarms (precision) vs misses (recall)** — the cost of missing a fault vs frequent false alarms. Also **detection latency**.

**其他常用方法**(可举): 预测模型残差(ARIMA/LSTM 预测误差大=异常)、**孤立森林(Isolation Forest)**、滑动窗口统计、季节性 ESD(S-H-ESD)、自编码器重建误差(呼应 13.1)、用于序列的专门模型。
**Other common methods:** forecast-model residuals (large ARIMA/LSTM errors = anomaly), **Isolation Forest**, sliding-window statistics, Seasonal Hybrid ESD, autoencoder reconstruction error (echoing 13.1), dedicated sequence models.

```
通用框架: 建模'正常'(分解STL/预测ARIMA等) → 算残差(实际-期望) → 残差超阈值=异常
异常类型: 点异常(单点突变,最常见) / 上下文异常(值不极端但当时反常) / 集体异常(一段子序列反常)
普通z-score坑: 均值/标准差被异常自己拉偏→标准差变大→阈值变松→漏检
稳健统计: 中位数代替均值, MAD代替标准差; 稳健z=0.6745(x-median)/MAD; 对异常不敏感(Twitter ESD基础)
评估: 异常极少(不平衡)→别用准确率; 用precision(防误报)/recall(防漏检)权衡 + 检测延迟
其他方法: 预测残差/孤立森林/自编码器重建误差/S-H-ESD
```

### 💡 面试速查 / Interview cheat-sheet
1. **框架**: 建模正常(分解/预测)→残差(实际-期望)→超阈值=异常。
   Framework: model normal → residual → threshold = anomaly.
2. **稳健统计**: 用中位数/MAD代替均值/标准差; 否则异常自己污染标尺(漏检)。
   Robust stats: median/MAD over mean/std; else anomalies corrupt the yardstick (misses).
3. **异常类型**: 点/上下文/集体。
   Types: point/contextual/collective.
4. **评估**: 别用准确率(极不平衡); 用precision/recall权衡+检测延迟。
   Eval: not accuracy (imbalanced); precision/recall trade-off + latency.
5. **其他方法**: 预测残差/孤立森林/自编码器重建误差/ESD。
   Others: forecast residuals/Isolation Forest/autoencoder/ESD.

### 下一节 / Next
**14.11 因果性检验(Part 14 收尾)**——预测看的是"相关", 但有时我们想知道"**X 是否真的影响/预测 Y**"。**Granger 因果检验**用 VAR 框架检验"加入 X 的历史, 能否显著改善对 Y 的预测"。我们会讲清它测的到底是什么(及它**不等于**真正的因果)。
**14.11 Granger Causality** (finale) — forecasting is about correlation, but sometimes we ask "**does X actually help predict Y**?" The **Granger causality test** uses VAR to check whether "adding X's history significantly improves predicting Y." We'll clarify what it really tests (and why it's **not** true causation).
